## Use Project Babel for inferencing using Turing NLG v2
- Load collection of prompts 
- Give to data preprocessing module to construct inference data 
- Point inferencing module to inference data and pre-trained (not fine-tuned) model

In [8]:
from azure.ml.babel import TuringULGV2Config
from azure.ml.babel import TuringULGV2ForConditionalGeneration
from azure.ml.babel import TuringULGV2Tokenizer

from azure.ml.babel import BabelAssistant
import torch

In [11]:
model_id = "tulg-v2-base-cased"

# load babel assistant to make it easy to access pretrained model weights
babel_assistant = BabelAssistant()

config = TuringULGV2Config.from_pretrained(model_id)

# to run on a CPU we disable fp16
device = "cpu"
config.extra_config["fs_args"]["fp16"] = False

# set config to use the decoder
config.use_decoder = True

tokenizer = TuringULGV2Tokenizer.from_pretrained(model_id, config=config)

with torch.no_grad():
    model = TuringULGV2ForConditionalGeneration.from_pretrained(
        model_id,
        config=config,
        babel_assistant=babel_assistant,
        ).eval()
model = model.to(device)

In [3]:
tokenizer.decode(tokenizer.eos_token_id)

'</s>'

In [14]:
# text to be translated, note multiple languages are supported
input_src = \
[
    ("Q: How large is your circle of friends?\n\nA: ", "en"),
    # ("My name is Alex. I am from Brazil, very nice to meet you.", "pt"),
    # ("My name is Amin. I am from the UK, very nice to meet you.", "ru"),
    # ("Je pense, donc je suis", "en"),
]

beam = 5
for src, tgt_lang in input_src:
    # lang = "__" + tgt_lang + "__"
    # features = [tokenizer.convert_tokens_to_ids(lang)]
    # features.extend(tokenizer.encode_plus(src, add_special_tokens=False, max_length=510, truncation=True)["input_ids"])
    features = tokenizer.encode_plus(src, add_special_tokens=False, max_length=510, truncation=True)["input_ids"]
    features.append(tokenizer.eos_token_id)

    pred_ids = model.generate(
        torch.LongTensor(features).unsqueeze(0).to(device),
        # decoder_start_token_id=tokenizer.eos_token_id,
        num_beams=beam,
        num_return_sequences=beam,
        do_sample=False, # do_sample not supported for tulg-v2
        # max_decode_length=80,
        temperature=0.8,
        max_length=256,
        )

    print(tokenizer.decode(pred_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False))

/anaconda/envs/babel/lib/python3.8/site-packages/azure/ml/babel/dependencies/zcode_transformers/generation_utils.py:1798: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  next_indices = next_tokens // vocab_size


Q: How large is your circle of friends? A:
